## ZEPTO CAPSTONE ##

In [ ]:
from bs4 import BeautifulSoup
import requests
import random
import pandas as pd
import sqlite3

BASE_URL = "https://books.toscrape.com/"

homepage = BeautifulSoup(requests.get(BASE_URL).text, "html.parser")

category_links = homepage.select("div.side_categories ul li ul li a")

categories = []
for link in random.sample(category_links,7):
    name = link.get_text(strip=True)
    url = BASE_URL + link["href"]
    categories.append((name, url))

all_books = []

for cat_name, cat_url in categories:
    next_url = cat_url
    count = 0

    while next_url:
        response = requests.get(next_url)
        response.encoding = "utf-8"
        soup = BeautifulSoup(response.text, "html.parser")
        

        for book in soup.select("article.product_pod"):
            title = book.h3.a["title"]
            price = book.select_one("p.price_color").get_text(strip=True)
            rating = book.select_one("p.star-rating")["class"][-1]
            availability = book.select_one("p.instock.availability").get_text(strip=True)

            all_books.append({
                "title": title,
                "price": price,
                "star_rating": rating,
                "availability": availability,
                "category": cat_name,
            })
            count += 1

        nextpage = soup.select_one("li.next a")
        next_url = next_url.rsplit("/", 1)[0] + "/" + nextpage["href"] if nextpage else None

    print(f"{cat_name}: {count} books")

print(f"\nTotal books scraped: {len(all_books)}")
for b in all_books:
    print(b)

Romance: 35 books
Add a comment: 67 books
Food and Drink: 30 books
Christian Fiction: 6 books
Health: 4 books
Politics: 3 books
Religion: 7 books

Total books scraped: 152
{'title': 'Chase Me (Paris Nights #2)', 'price': '£25.27', 'star_rating': 'Five', 'availability': 'In stock', 'category': 'Romance'}
{'title': 'Black Dust', 'price': '£34.53', 'star_rating': 'Five', 'availability': 'In stock', 'category': 'Romance'}
{'title': 'Her Backup Boyfriend (The Sorensen Family #1)', 'price': '£33.97', 'star_rating': 'One', 'availability': 'In stock', 'category': 'Romance'}
{'title': 'First and First (Five Boroughs #3)', 'price': '£15.97', 'star_rating': 'Four', 'availability': 'In stock', 'category': 'Romance'}
{'title': 'Fifty Shades Darker (Fifty Shades #2)', 'price': '£21.96', 'star_rating': 'One', 'availability': 'In stock', 'category': 'Romance'}
{'title': 'The Wedding Dress', 'price': '£24.12', 'star_rating': 'One', 'availability': 'In stock', 'category': 'Romance'}
{'title': 'Suddenly 

In [ ]:

df = pd.DataFrame(all_books)

print(df.shape)
df

(152, 5)


,title,price,star_rating,availability,category
0,Chase Me (Paris Nights #2),£25.27,Five,In stock,Romance
1,Black Dust,£34.53,Five,In stock,Romance
2,Her Backup Boyfriend (The Sorensen Family #1),£33.97,One,In stock,Romance
3,First and First (Five Boroughs #3),£15.97,Four,In stock,Romance
4,Fifty Shades Darker (Fifty Shades #2),£21.96,One,In stock,Romance
...,...,...,...,...,...
147,God: The Most Unpleasant Character in All Fiction,£30.03,Five,In stock,Religion
148,The Book of Mormon,£24.57,Three,In stock,Religion
149,"A History of God: The 4,000-Year Quest of Juda...",£27.62,One,In stock,Religion
150,The Bhagavad Gita,£57.49,Three,In stock,Religion


In [5]:
df['price_gbp']=df['price'].str.replace(r"[^\d.]", "", regex=True).astype(float)


rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}

df["star_rating"] = df["star_rating"].map(rating_map)

In [7]:
df['in_stock']=df['availability'].str.contains('In stock',case=False,na=False)

In [8]:
price_inr=[]
for price in df['price_gbp']:
    price_value= price * 105.50
    price_inr.append(price_value)

df['price_inr']=price_inr

In [9]:
df

,title,price,star_rating,availability,category,price_gbp,in_stock,price_inr
0,Chase Me (Paris Nights #2),£25.27,5,In stock,Romance,25.27,True,2665.985
1,Black Dust,£34.53,5,In stock,Romance,34.53,True,3642.915
2,Her Backup Boyfriend (The Sorensen Family #1),£33.97,1,In stock,Romance,33.97,True,3583.835
3,First and First (Five Boroughs #3),£15.97,4,In stock,Romance,15.97,True,1684.835
4,Fifty Shades Darker (Fifty Shades #2),£21.96,1,In stock,Romance,21.96,True,2316.780
...,...,...,...,...,...,...,...,...
147,God: The Most Unpleasant Character in All Fiction,£30.03,5,In stock,Religion,30.03,True,3168.165
148,The Book of Mormon,£24.57,3,In stock,Religion,24.57,True,2592.135
149,"A History of God: The 4,000-Year Quest of Juda...",£27.62,1,In stock,Religion,27.62,True,2913.910
150,The Bhagavad Gita,£57.49,3,In stock,Religion,57.49,True,6065.195


In [10]:
print(df.columns)

Index(['title', 'price', 'star_rating', 'availability', 'category',
       'price_gbp', 'in_stock', 'price_inr'],
      dtype='str')


In [ ]:


conn = sqlite3.connect("books.db")
cur = conn.cursor()

cur.execute("PRAGMA foreign_keys = ON")

cur.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

cur.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    price REAL,
    star_rating REAL,
    availability TEXT,
    price_gbp REAL,
    in_stock INTEGER,
    price_inr REAL,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

for cat in df["category"].dropna().unique():
    cur.execute(
        "INSERT INTO categories (category_name) VALUES (?)",
        (cat,)
    )

cur.execute("""
    SELECT category_id, category_name
    FROM categories
""")

category_list = {
    name: cid
    for cid, name in cur.fetchall()
}

for _, row in df.iterrows():

    cur.execute("""
        INSERT INTO books (
            title,
            price,
            star_rating,
            availability,
            price_gbp,
            in_stock,
            price_inr,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price"],
        row["star_rating"],
        row["availability"],
        row["price_gbp"],
        row["in_stock"],
        row["price_inr"],
        category_list[row["category"]]
    ))

conn.commit()

print("Database created and all data inserted successfully.")

Database created and all data inserted successfully.


In [ ]:

# Query 1: SELECT + WHERE
query1 = """
SELECT title, price_inr, star_rating
FROM books
WHERE star_rating >= 4
"""

output1 = pd.read_sql(query1, conn)


# Query 2: ORDER BY
query2 = """
SELECT title, price_inr
FROM books
ORDER BY price_inr DESC
"""

output2 = pd.read_sql(query2, conn)


# Query 3: LIMIT
query3 = """
SELECT title, star_rating
FROM books
ORDER BY star_rating DESC
LIMIT 10
"""

output3 = pd.read_sql(query3, conn)


# Query 4: DISTINCT
query4 = """
SELECT DISTINCT category_name
FROM categories
ORDER BY category_name
"""

output4 = pd.read_sql(query4, conn)


# Query 5: BETWEEN
query5 = """
SELECT title, price_inr, star_rating
FROM books
WHERE price_inr BETWEEN 500 AND 1500
ORDER BY price_inr
"""

output5 = pd.read_sql(query5, conn)


# Query 6: JOIN
query6 = """
SELECT
    b.title,
    b.star_rating,
    b.price_inr,
    c.category_name
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.star_rating DESC
LIMIT 10
"""

output6 = pd.read_sql(query6, conn)

In [ ]:


queries_and_outputs = [
    ("QUERY 1 - SELECT + WHERE", query1, output1),
    ("QUERY 2 - ORDER BY", query2, output2),
    ("QUERY 3 - LIMIT", query3, output3),
    ("QUERY 4 - DISTINCT", query4, output4),
    ("QUERY 5 - BETWEEN", query5, output5),
    ("QUERY 6 - JOIN", query6, output6)
]

for name, query, output in queries_and_outputs:

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("\nSQL:")
    print(query)

    print("\nOUTPUT:")
    print(output.to_string(index=False))


QUERY 1 - SELECT + WHERE

SQL:

SELECT title, price_inr, star_rating
FROM books
WHERE star_rating >= 4


OUTPUT:
                                                                                                                                                 title  price_inr  star_rating
                                                                                                                            Chase Me (Paris Nights #2)   2665.985          5.0
                                                                                                                                            Black Dust   3642.915          5.0
                                                                                                                    First and First (Five Boroughs #3)   1684.835          4.0
                                                                                                                              Something More Than This   1713.320          4.0
           

In [ ]:

sql_join_result = pd.read_sql(query6, conn)

print("SQL JOIN RESULT:")
print(sql_join_result.to_string(index=False))

SQL JOIN RESULT:
                                                  title  star_rating  price_inr category_name
                             Chase Me (Paris Nights #2)          5.0   2665.985       Romance
                                             Black Dust          5.0   3642.915       Romance
       A Gentleman's Position (Society of Gentlemen #3)          5.0   1556.125       Romance
                        Deep Under (Walker Security #1)          5.0   4967.995       Romance
                                         Modern Romance          5.0   2981.430 Add a comment
                  The White Queen (The Cousins' War #1)          5.0   2733.505 Add a comment
                                   The Song of Achilles          5.0   3945.700 Add a comment
                                          Without Shame          5.0   5092.485 Add a comment
Team of Rivals: The Political Genius of Abraham Lincoln          5.0   2122.660 Add a comment
                        Good in Bed (Cannie

In [ ]:

books_df = pd.read_sql("SELECT * FROM books", conn)
categories_df = pd.read_sql("SELECT * FROM categories", conn)


merge_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)


merge_result = merge_result[
    [
        "title",
        "star_rating",
        "price_inr",
        "category_name"
    ]
]

merge_result = (
    merge_result
    .sort_values(
        by="star_rating",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

print("PANDAS MERGE RESULT:")
print(merge_result.to_string(index=False))

PANDAS MERGE RESULT:
                                                                                  title  star_rating  price_inr     category_name
                                                             Chase Me (Paris Nights #2)          5.0   2665.985           Romance
                                                                             Black Dust          5.0   3642.915           Romance
                                       A Gentleman's Position (Society of Gentlemen #3)          5.0   1556.125           Romance
Naturally Lean: 125 Nourishing Gluten-Free, Plant-Based Recipes--All Under 300 Calories          5.0   1200.590    Food and Drink
                  Mexican Today: New and Rediscovered Recipes for Contemporary Kitchens          5.0   2628.005    Food and Drink
                        10-Day Green Smoothie Cleanse: Lose Up to 15 Pounds in 10 Days!          5.0   5244.405            Health
                                                   Shadows of the Pas

In [ ]:

sql_result = sql_join_result.copy()
pandas_result = merge_result.copy()

sql_result["Source"] = "SQL JOIN"
pandas_result["Source"] = "Pandas merge()"


side_by_side = pd.concat(
    [
        sql_result.add_suffix("_SQL"),
        pandas_result.add_suffix("_Pandas")
    ],
    axis=1
)

print("\n" + "=" * 100)
print("SQL JOIN vs pandas.merge()")
print("=" * 100)

print(side_by_side.to_string(index=False))


SQL JOIN vs pandas.merge()
                                              title_SQL  star_rating_SQL  price_inr_SQL category_name_SQL Source_SQL                                                                            title_Pandas  star_rating_Pandas  price_inr_Pandas category_name_Pandas  Source_Pandas
                             Chase Me (Paris Nights #2)              5.0       2665.985           Romance   SQL JOIN                                                              Chase Me (Paris Nights #2)                 5.0          2665.985              Romance Pandas merge()
                                             Black Dust              5.0       3642.915           Romance   SQL JOIN                                                                              Black Dust                 5.0          3642.915              Romance Pandas merge()
       A Gentleman's Position (Society of Gentlemen #3)              5.0       1556.125           Romance   SQL JOIN                   